# 검수 코드 동기화 검사

같은 전처리(`step1_clean` / `step2_redefine` / `step3_fill` / `_is_batting` / 판정 상수…)가
여러 파일에 복사돼 있다. 갈라져도 **에러가 안 나고 결과만 조용히 틀리기** 때문에 눈으로는
못 잡는다 — 실제로 `_is_batting` 이 `루공과` 처리에서 갈라져 있던 걸 이 검사로 찾았다.

| 파일 | 역할 |
|---|---|
| `2-3. 통합 체크 (아웃·점수 + 주자상황).ipynb` | 평소 돌리는 통합본 |
| `2. 아웃카운트 및 점수 체크.ipynb` | 아웃/점수를 inning_summary 와 대조 |
| `3. 주자상황 체크.ipynb` | 주자 시뮬 + batter/pitcher box 대조 |
| `scripts/pipeline_local.py` | 3번의 로컬 포팅본(디버깅 레퍼런스) |

**노트북을 고쳤으면 나머지도 같이 고치고 이 셀을 한 번 돌릴 것.**
없는 파일은 그냥 건너뛴다(Colab 에는 `local_mirror` 가 없다 — 그래도 노트북끼리는 대조된다).


In [ ]:
# ── 이 노트북이 있는 폴더를 기준으로 찾는다.
#    Colab 이면 아래 BASE 를 Drive 경로로 바꿀 것:
#    BASE = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
import ast
import difflib
import json
import os

BASE = os.getcwd()

SOURCES = [
    ("통합",    "2-3. 통합 체크 (아웃·점수 + 주자상황).ipynb"),
    ("노트북2", "2. 아웃카운트 및 점수 체크.ipynb"),
    ("노트북3", "3. 주자상황 체크.ipynb"),
    ("local",   "scripts/pipeline_local.py"),
]

# 이름이 같으면 내용도 같아야 하는 대상. 함수는 이름으로, 상수는 '대문자 이름의 최상위
# 대입'을 전부 본다. 예전에는 상수를 손으로 목록에 적었는데, 목록에 없는 상수가 갈라지면
# 돌려도 못 잡았다 — 2026-09-12 에 NT_ERRORS·야선(R_RESULTS) 이 3번 노트북·pipeline_local
# 에만 들어가고 통합본·2번에서 빠진 채 한 달 가까이 남았다.
# 경로·연도처럼 원래 달라야 하는 것만 뺀다.
ENV_SPECIFIC = {"BASE", "BASE_DIR", "YEAR", "MISBAT_ORDER_PATH"}


def is_constant(name):
    core = name.lstrip("_")
    return (any(ch.isalpha() for ch in core) and core.upper() == core
            and name not in ENV_SPECIFIC)

# 실행 진입점은 환경마다 달라야 정상이다(Colab 은 Drive 경로·연도 루프, 로컬 포팅본은
# 미러 폴더를 훑는다). 공유 로직이 아니므로 비교 대상에서 뺀다.
EXCLUDE = {"run_all_pipeline", "run_tracking_pipeline", "run_check_pipeline", "main"}

# 콘솔 인코딩 때문에 일부러 갈라놓은 표기. pipeline_local 은 Windows 콘솔(cp949)에서
# 돌아서 '→'/'⚠' 를 찍으면 UnicodeEncodeError 로 죽는다 — 고칠 차이가 아니라 유지할 차이다.
_COSMETIC = [("→", "->"), ("⚠ ", ""), ("⚠", "")]


def behavior_key(src):
    """비교용 정규화 — 동작이 같으면 같은 값이 나오게 만든다.

    AST 로 비교하므로 주석·빈 줄·정렬 공백은 애초에 빠지고, 문자열 상수의 콘솔 기호만
    따로 맞춰준다. 소스를 그대로 비교하면 무해한 차이까지 매번 실패로 떠서, 정작 진짜
    드리프트가 났을 때 아무도 안 본다."""
    tree = ast.parse(src)
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            v = node.value
            for a, b in _COSMETIC:
                v = v.replace(a, b)
            node.value = v
    return ast.dump(tree)


def _chunks(path):
    """비교 대상 소스 조각. 노트북은 코드 셀마다 따로 파싱한다 — 셀을 이어 붙이면
    한 셀의 문법 오류가 전체 파싱을 막고, 셀 경계가 어긋나도 티가 안 난다."""
    if path.endswith(".ipynb"):
        with open(path, encoding="utf-8") as f:
            nb = json.load(f)
        return ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    with open(path, encoding="utf-8") as f:
        return [f.read()]


def collect(path):
    """{이름: 소스}. 최상위 함수와 대문자 이름의 상수(is_constant)를 모은다."""
    out = {}
    for chunk in _chunks(path):
        try:
            tree = ast.parse(chunk)
        except SyntaxError:
            continue          # Colab 매직(%…) 이 섞인 셀 등은 건너뛴다
        for node in tree.body:
            name = None
            if isinstance(node, ast.FunctionDef):
                name = node.name
            elif (isinstance(node, ast.Assign) and len(node.targets) == 1
                  and isinstance(node.targets[0], ast.Name)
                  and is_constant(node.targets[0].id)):
                name = node.targets[0].id
            if name is None:
                continue
            src = ast.get_source_segment(chunk, node)
            if src is not None:
                out[name] = src   # 같은 이름이 여러 셀에 있으면 마지막 정의가 유효하다
    return out


def check_sync(show_diff=True):
    tables = []
    for tag, rel in SOURCES:
        path = os.path.join(BASE, rel)
        if not os.path.exists(path):
            print(f"  · 건너뜀({tag}): {rel} 없음")
            continue
        tables.append((tag, collect(path)))

    if len(tables) < 2:
        print("\n비교할 파일이 2개 미만입니다 — BASE 경로를 확인하세요:", BASE)
        return 2

    problems = 0
    for i in range(len(tables)):
        for j in range(i + 1, len(tables)):
            tag_a, a = tables[i]
            tag_b, b = tables[j]
            shared = sorted((set(a) & set(b)) - EXCLUDE)
            names = [n for n in shared if behavior_key(a[n]) != behavior_key(b[n])]
            print(f"[{tag_a} ↔ {tag_b}] 공통 {len(shared)}개 중 다름 {len(names)}개"
                  + (f" — {', '.join(names)}" if names else ""))
            problems += len(names)
            if not show_diff:
                continue
            for n in names:
                print(f"\n----- {n} ({tag_a} vs {tag_b}) -----")
                for line in difflib.unified_diff(
                    a[n].splitlines(), b[n].splitlines(), tag_a, tag_b, lineterm="", n=1
                ):
                    print("  " + line)

    print()
    if problems:
        print(f"✗ 갈라진 정의 {problems}건 — 맞추세요.")
    else:
        print("✓ 공유 정의가 모두 일치합니다.")
    return problems


check_sync()
